# 3. Planeación de horarios escolares

El encargado de la planificación académica en el Departamento de Ingenierı́a de la Universidad Estatal intenta resolver el ((problema de horarios académicos)). El objetivo es **asociar aulas y horas a las asignaturas** de un programa académico **dividido en dos cursos**.

Se cuenta con **tres aulas** y un total de **cinco horas** disponibles para programar las clases de **ocho asignaturas distintas**. Estas asignaturas se reparten entre **dos docentes** y entre **dos cursos académicos**. El conjunto completo de asignaturas es

$$\text{Materias} = \{M_1, M_2, M_3 , M_4 , M_5 , M_6, M_7, M_8 \}$$

* El **Docente** 1 imparte tres *asignaturas*: $M_1$ , $M_2$ y $M_8$.
* El **Docente** 2 imparte cinco *asignaturas*: $M_3$, $M_4$, $M_5$ , $M_6$ y $M_7$ .

* El primer **curso** toma las *asignaturas* $M_1$, $M_2$, $M_3$ y $M_4$.
* El segundo **curso** toma las *asignaturas* $M_5$ , $M_6$ , $M_7$ y $M_8$ .

Es esencial tener en cuenta las siguientes condiciones:

* Cada **docente** debe impartir todas las *asignaturas* que tiene asignadas. ✅ la 1
* Cada **docente** no puede impartir más de una *asignatura* en una misma hora.✅
* Cada *asignatura* debe impartirse una sola vez. ✅
* En cada **aula** y en cada **hora** se permite impartir a lo más una *asignatura*. ✅
* Se permite impartir a lo más una *asignatura* de cada **curso** en una misma **hora**.

**a)** Construya un modelo de programación lineal entera **que asocie las tres aulas y las cinco horas a las ocho asignaturas**, respetando todas las condiciones anteriores y logrando un horario compacto.

----

## Datos y supuestos  
* Asociar: Asignaturas - Aulas - Horas  
* 3 aulas 
* 5 horas 
* 8 asignatuas 

In [10]:
from docplex.mp.model import Model
import numpy as np

mdl = Model(name='Planeacion de horario')

## Formulación matemática 
Cada asignatura debesele asignar aula y hora
### Variables de decisión
Cada asignatura $i$ debesele asignar aula $j$, hora $k$ 
$$x_{ijk} =
\begin{cases}
	1 & \text{si la asignatura $i$ es impartida en el aula $j$ a la hora $k$}
	\\
	0 & \text{si no}
\end{cases}$$
donde $$i\ \epsilon\ \{0, 1, 2, 3, 4, 5, 6, 7\} = \text{Materia }\{M_1, M_2, M_3, M_4, M_5, M_6, M_7, M_8\}$$ ,
$$j\ \epsilon\ \{0, 1, 2\} = \text{Aula }\{1, 2, 3\}$$
y 
$$k\ \epsilon\ \{0, 1, 2, 3, 4\} = \text{Bloque }\{1, 2, 3, 4, 5\}$$

In [11]:
materias_i = range(8) 
aulas_j = range(3) 
horas_k = range(5)

In [12]:
x = mdl.binary_var_dict(
    [(i, j, k) for i in materias_i for j in aulas_j for k in horas_k], 
    name='x'
)

x

{(0, 0, 0): docplex.mp.Var(type=B,name='x_0_0_0'),
 (0, 0, 1): docplex.mp.Var(type=B,name='x_0_0_1'),
 (0, 0, 2): docplex.mp.Var(type=B,name='x_0_0_2'),
 (0, 0, 3): docplex.mp.Var(type=B,name='x_0_0_3'),
 (0, 0, 4): docplex.mp.Var(type=B,name='x_0_0_4'),
 (0, 1, 0): docplex.mp.Var(type=B,name='x_0_1_0'),
 (0, 1, 1): docplex.mp.Var(type=B,name='x_0_1_1'),
 (0, 1, 2): docplex.mp.Var(type=B,name='x_0_1_2'),
 (0, 1, 3): docplex.mp.Var(type=B,name='x_0_1_3'),
 (0, 1, 4): docplex.mp.Var(type=B,name='x_0_1_4'),
 (0, 2, 0): docplex.mp.Var(type=B,name='x_0_2_0'),
 (0, 2, 1): docplex.mp.Var(type=B,name='x_0_2_1'),
 (0, 2, 2): docplex.mp.Var(type=B,name='x_0_2_2'),
 (0, 2, 3): docplex.mp.Var(type=B,name='x_0_2_3'),
 (0, 2, 4): docplex.mp.Var(type=B,name='x_0_2_4'),
 (1, 0, 0): docplex.mp.Var(type=B,name='x_1_0_0'),
 (1, 0, 1): docplex.mp.Var(type=B,name='x_1_0_1'),
 (1, 0, 2): docplex.mp.Var(type=B,name='x_1_0_2'),
 (1, 0, 3): docplex.mp.Var(type=B,name='x_1_0_3'),
 (1, 0, 4): docplex.mp.Var(type

### Restricciones

In [13]:
# grupos de materias por docente
docente_1 = [0, 1, 7]        # M1, M2, M8
docente_2 = [2, 3, 4, 5, 6]  # M3, M4, M5, M6, M7

curso_1 = [0, 1, 2, 3]       # M1, M2, M3, M4
curso_2 = [4, 5, 6, 7]       # M5, M6, M7, M8

**Restricciones 1** 
* Cada *asignatura* $i$ debe impartirse una sola vez.
* Equivalente a "Cada **docente** debe impartir todas las *asignaturas* que tiene asignadas."
* Dejamos $i$ fijo y sumamos sobre los demas indices. 
$$ \sum_{j}\sum_{k}x_{ijk} = 1 \ \ \forall \ i$$

In [14]:
mdl.add_constraints(
    mdl.sum(x[i, j, k] for j in aulas_j for k in horas_k) == 1
    for i in materias_i
)

[docplex.mp.LinearConstraint[](x_0_0_0+x_0_0_1+x_0_0_2+x_0_0_3+x_0_0_4+x_0_1_0+x_0_1_1+x_0_1_2+x_0_1_3+x_0_1_4+x_0_2_0+x_0_2_1+x_0_2_2+x_0_2_3+x_0_2_4,EQ,1),
 docplex.mp.LinearConstraint[](x_1_0_0+x_1_0_1+x_1_0_2+x_1_0_3+x_1_0_4+x_1_1_0+x_1_1_1+x_1_1_2+x_1_1_3+x_1_1_4+x_1_2_0+x_1_2_1+x_1_2_2+x_1_2_3+x_1_2_4,EQ,1),
 docplex.mp.LinearConstraint[](x_2_0_0+x_2_0_1+x_2_0_2+x_2_0_3+x_2_0_4+x_2_1_0+x_2_1_1+x_2_1_2+x_2_1_3+x_2_1_4+x_2_2_0+x_2_2_1+x_2_2_2+x_2_2_3+x_2_2_4,EQ,1),
 docplex.mp.LinearConstraint[](x_3_0_0+x_3_0_1+x_3_0_2+x_3_0_3+x_3_0_4+x_3_1_0+x_3_1_1+x_3_1_2+x_3_1_3+x_3_1_4+x_3_2_0+x_3_2_1+x_3_2_2+x_3_2_3+x_3_2_4,EQ,1),
 docplex.mp.LinearConstraint[](x_4_0_0+x_4_0_1+x_4_0_2+x_4_0_3+x_4_0_4+x_4_1_0+x_4_1_1+x_4_1_2+x_4_1_3+x_4_1_4+x_4_2_0+x_4_2_1+x_4_2_2+x_4_2_3+x_4_2_4,EQ,1),
 docplex.mp.LinearConstraint[](x_5_0_0+x_5_0_1+x_5_0_2+x_5_0_3+x_5_0_4+x_5_1_0+x_5_1_1+x_5_1_2+x_5_1_3+x_5_1_4+x_5_2_0+x_5_2_1+x_5_2_2+x_5_2_3+x_5_2_4,EQ,1),
 docplex.mp.LinearConstraint[](x_6_0_0+x_6_0_1+x_6_0

**Restricciones 2** 
* En cada **aula** $j$ y en cada **hora** $k$ se permite impartir ==a lo más== una *asignatura* $i$.
* Dejamos $j$ y $k$ fijos y sumamos sobre la asignatura $i$. 
* Para cada aula $j$ y hora $k$ específicos, solo debe haber una materia -> suma de materias = 1 para cada aula y hora especificos. 

$$ \sum_{i}x_{ijk} \leq 1 \ \ \forall \ j, k$$


In [15]:
mdl.add_constraints(mdl.sum(x[i, j, k] for i in materias_i) <= 1
                    for j in aulas_j 
                    for k in horas_k 
                    )

[docplex.mp.LinearConstraint[](x_0_0_0+x_1_0_0+x_2_0_0+x_3_0_0+x_4_0_0+x_5_0_0+x_6_0_0+x_7_0_0,LE,1),
 docplex.mp.LinearConstraint[](x_0_0_1+x_1_0_1+x_2_0_1+x_3_0_1+x_4_0_1+x_5_0_1+x_6_0_1+x_7_0_1,LE,1),
 docplex.mp.LinearConstraint[](x_0_0_2+x_1_0_2+x_2_0_2+x_3_0_2+x_4_0_2+x_5_0_2+x_6_0_2+x_7_0_2,LE,1),
 docplex.mp.LinearConstraint[](x_0_0_3+x_1_0_3+x_2_0_3+x_3_0_3+x_4_0_3+x_5_0_3+x_6_0_3+x_7_0_3,LE,1),
 docplex.mp.LinearConstraint[](x_0_0_4+x_1_0_4+x_2_0_4+x_3_0_4+x_4_0_4+x_5_0_4+x_6_0_4+x_7_0_4,LE,1),
 docplex.mp.LinearConstraint[](x_0_1_0+x_1_1_0+x_2_1_0+x_3_1_0+x_4_1_0+x_5_1_0+x_6_1_0+x_7_1_0,LE,1),
 docplex.mp.LinearConstraint[](x_0_1_1+x_1_1_1+x_2_1_1+x_3_1_1+x_4_1_1+x_5_1_1+x_6_1_1+x_7_1_1,LE,1),
 docplex.mp.LinearConstraint[](x_0_1_2+x_1_1_2+x_2_1_2+x_3_1_2+x_4_1_2+x_5_1_2+x_6_1_2+x_7_1_2,LE,1),
 docplex.mp.LinearConstraint[](x_0_1_3+x_1_1_3+x_2_1_3+x_3_1_3+x_4_1_3+x_5_1_3+x_6_1_3+x_7_1_3,LE,1),
 docplex.mp.LinearConstraint[](x_0_1_4+x_1_1_4+x_2_1_4+x_3_1_4+x_4_1_4+x_5_1_4+x_6

**Restricciones 3** 

Cada **docente** no puede impartir más de una *asignatura* $i$ en una misma hora $k$.

**3.1** - El Docente 1 imparte tres asignaturas: $M_1$ , $M_2$ y $M_8$.

$$\sum_{i \epsilon \{0, 1, 7\}}\sum_{j} x_{ijk} \leq 1 \ \ \forall \ \ k$$

**3.2** - El Docente 2 imparte cinco asignaturas: $M_3$, $M_4$, $M_5$ , $M_6$ y $M_7$ .

$$\sum_{i \epsilon \{2, 3, 4, 5, 6\}}\sum_{j} x_{ijk} \leq 1 \ \ \forall \ \ k$$


In [16]:
for k in horas_k:
    mdl.add_constraint(mdl.sum(x[i, j, k] for i in docente_1 for j in aulas_j) <= 1)
    mdl.add_constraint(mdl.sum(x[i, j, k] for i in docente_2 for j in aulas_j) <= 1)

**Restricciones 4** 
Se permite impartir a lo más una *asignatura* $i$ de cada **curso** en una misma **hora** $k$.

**4.1** - El primer **curso** toma las *asignaturas* $M_1$, $M_2$, $M_3$ y $M_4$.

$$\sum_{i \epsilon \{0, 1, 2, 3\}}\sum_{j} x_{ijk} \leq 1 \ \ \forall \ \ k$$

**4.2** El segundo **curso** toma las *asignaturas* $M_5$ , $M_6$ , $M_7$ y $M_8$ .

$$\sum_{i \epsilon \{4, 5, 6, 7\}}\sum_{j} x_{ijk} \leq 1 \ \ \forall \ \ k$$

In [17]:
for k in horas_k:
    mdl.add_constraint(mdl.sum(x[i, j, k] for i in curso_1 for j in aulas_j) <= 1)
    mdl.add_constraint(mdl.sum(x[i, j, k] for i in curso_2 for j in aulas_j) <= 1)

## Función objetivo 
* Minimizar el tiempo total empleado
* un **"horario compacto"** busca programar las clases en los **primeros bloques de horas posibles** (evitando huecos/ventanas y terminando la jornada lo antes posible).
* Para incentivar al solucionador a usar los bloques de horas más tempranos ($k=0, 1, 2 \dots$), podemos ponderar el uso de la hora $k$ con un costo incremental proporcional a $k$ (o $k+1$):
$$\min z = \sum_{i=0}^{7} \sum_{j=0}^{2} \sum_{k=0}^{4} (k + 1) \cdot x_{ijk}$$


In [18]:
mdl.minimize(mdl.sum((k + 1) * x[i, j, k] for i in materias_i for j in aulas_j for k in horas_k))

In [19]:
solution = mdl.solve(log_output=True)

Version identifier: 22.2.0.0 | 2026-05-04 | 5dff98e58
CPXPARAM_Read_DataCheck                          1
Found incumbent of value 21.000000 after 0.01 sec. (0.01 ticks)
Tried aggregator 1 time.
Reduced MIP has 43 rows, 120 columns, and 480 nonzeros.
Reduced MIP has 120 binaries, 0 generals, 0 SOSs, and 0 indicators.
Presolve time = 0.01 sec. (0.26 ticks)
Probing time = 0.00 sec. (0.20 ticks)
Tried aggregator 1 time.
Detecting symmetries...
Reduced MIP has 43 rows, 120 columns, and 480 nonzeros.
Reduced MIP has 120 binaries, 0 generals, 0 SOSs, and 0 indicators.
Presolve time = 0.01 sec. (0.35 ticks)
Probing time = 0.00 sec. (0.20 ticks)
Clique table members: 43.
MIP emphasis: balance optimality and feasibility.
MIP search method: dynamic search.
Parallel mode: deterministic, using up to 4 threads.
Root relaxation solution time = 0.00 sec. (0.15 ticks)

        Nodes                                         Cuts/
   Node  Left     Objective  IInf  Best Integer    Best Bound    ItCnt     

???

In [20]:
if solution:
    print(f"Valor óptimo de la función objetivo: {solution.objective_value}\n")
    print("=================== HORARIO ESCOLAR OPTIMIZADO ===================")
    for k in horas_k:
        print(f"\n--- Bloque {k + 1} ---")
        clases_en_bloque = 0
        for j in aulas_j:
            for i in materias_i:
                if x[i, j, k].solution_value > 0.5:
                    print(f"  Aula {j + 1}: Materia M{i + 1}")
                    clases_en_bloque += 1
        if clases_en_bloque == 0:
            print("  (Sin clases programadas)")

Valor óptimo de la función objetivo: 21.0

=================== HORARIO ESCOLAR OPTIMIZADO ===================

--- Bloque 1 ---
  Aula 1: Materia M1
  Aula 2: Materia M6

--- Bloque 2 ---
  Aula 1: Materia M2
  Aula 2: Materia M7

--- Bloque 3 ---
  Aula 1: Materia M3
  Aula 2: Materia M8

--- Bloque 4 ---
  Aula 1: Materia M4

--- Bloque 5 ---
  Aula 1: Materia M5
